# EXP-005 — Random Forest mặc định

- **Ngày chạy lại** 18/11/2026
- **Trạng thái** Lần 1 dùng kết quả G3 đã lưu. Lần 2 đã chạy và xác minh đủ 30 fold.

## Lần 1 — Kết quả lịch sử G3-16/11/2026

Lần 1 dùng lại điểm Random Forest mặc định đã có từ G3. Mỗi bài toán có 15 lượt kiểm tra, mỗi lượt giữ riêng một người làm tập test. Kết quả này làm mốc so sánh với Lần 2.

Điểm trung bình trên 15 người cho mỗi bài toán.

| Bài toán | Macro-F1 | Balanced Accuracy |
|---|---:|---:|
| Nhị phân | 0,8735 | 0,8865 |
| Ba lớp | 0,6480 | 0,6700 |

Xem [điểm RF theo từng người](../../experiments/01_baselines/EXP-005_RF/results.csv) và [báo cáo tổng hợp G3](../../docs/G3_TONG_HOP.md).

## Lần 2 — Kiểm tra chạy lại và đối chiếu kết quả

Random Forest mặc định được chạy lại với 100 cây, seed 42, cùng bộ physiological68 và cách chia LOSO. Lượt này có hồ sơ xác minh riêng trong `outputs/exp005_rf_full/`.

## Đối chiếu

Ghép điểm theo bài toán và người test. Ghép dự đoán theo bài toán, người test và mã cửa sổ. Chênh lệch được tính bằng Lần 2 trừ Lần 1. Ngưỡng so sánh điểm là `1e-10`.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

here = Path.cwd()
repo = next(path for path in (here, *here.parents) if (path / "scripts/run_g3_baselines.py").is_file())
experiment = repo / "experiments/01_baselines/EXP-005_RF"
output = repo / "outputs/exp005_rf_full"
verification = json.loads((output / "verification.json").read_text(encoding="utf-8"))
assert verification["status"] == "passed" and verification["models"] == ["rf"]
assert verification["n_folds"] == 30 and verification["n_predictions"] == 12378

keys = ["task", "subject_id"]
metrics = ["macro_f1", "balanced_accuracy"]
g3 = pd.read_csv(experiment / "results.csv")
rerun = pd.read_csv(output / "verified_metrics_by_subject.csv")
points = g3[keys + metrics].merge(
    rerun[keys + metrics], on=keys, suffixes=("_g3", "_rerun"), validate="one_to_one"
)
assert len(g3) == len(rerun) == len(points) == 30

for metric in metrics:
    points[f"{metric}_delta"] = points[f"{metric}_rerun"] - points[f"{metric}_g3"]

columns = [f"{metric}_{part}" for metric in metrics for part in ("g3", "rerun", "delta")]
display(points.groupby("task")[columns].mean().round(12))
delta = points[[f"{metric}_delta" for metric in metrics]]
changed = ~np.isclose(delta, 0, atol=1e-10, rtol=0).all(axis=1)
print(f"Fold có điểm khác: {changed.sum()}/{len(points)}")
print(f"Chênh lệch tuyệt đối lớn nhất: {delta.abs().max().max():.3g}")
if changed.any():
    display(points.loc[changed])


In [ ]:
prediction_keys = ["task", "subject_id", "window_id"]
g3_predictions_path = repo / "outputs/g3_group_predictions_six_models_20261108/predictions.csv"
if g3_predictions_path.is_file():
    g3_predictions = pd.read_csv(g3_predictions_path).query("model == 'rf'")
    rerun_predictions = pd.read_csv(output / "predictions.csv")
    columns = prediction_keys + ["y_true", "y_pred"]
    predictions = g3_predictions[columns].merge(
        rerun_predictions[columns], on=prediction_keys,
        suffixes=("_g3", "_rerun"), validate="one_to_one"
    )
    assert len(predictions) == len(g3_predictions) == len(rerun_predictions) == 12378
    assert predictions["y_true_g3"].eq(predictions["y_true_rerun"]).all()
    changed_predictions = predictions["y_pred_g3"].ne(predictions["y_pred_rerun"])
    print(f"Dự đoán thay đổi: {changed_predictions.sum()}/{len(predictions)}")
else:
    print("Thiếu bảng dự đoán chi tiết G3 để đối chiếu từng cửa sổ.")


## Nhận xét

Ở lượt chạy lại đã xác minh, có **0/30 fold** thay đổi Macro-F1 hoặc Balanced Accuracy theo ngưỡng đối chiếu và **0/12.378 dự đoán** đổi nhãn. Hai lượt được lưu ở các nguồn khác nhau. Lần 1 là hồ sơ G3, còn Lần 2 là lần huấn luyện lại của EXP-005.

**Hai metric khớp kết quả G3 ở đủ 30 fold.**
